In [1]:
import requests
from settings import get_settings


def get_playlists(access_token: str) -> list[dict]:
    """Fetch all playlists for the authenticated user.

    Returns a list of playlist dictionaries.

    Args:
        access_token (str): The access token for the authenticated user.

    Returns:
        list: A list of playlist dictionaries.
    """
    headers = {"Authorization": f"Bearer {access_token}"}
    url = "https://api.spotify.com/v1/me/playlists"
    playlists = []

    while url:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch playlists: {response.status_code}")

        data = response.json()
        playlists.extend(data.get("items", []))
        url = data.get("next")  # Move to next page if available

    return playlists


def get_playlist_tracks(tracks_url: str, access_token: str) -> list[dict]:
    """Fetch tracks from a specific playlist using its tracks URL.

    Returns a list of track dictionaries.

    Args:
        tracks_url (str): The URL of the playlist's tracks.
        access_token (str): The access token for the authenticated user.

    Returns:
        list: A list of track dictionaries.
    """
    headers = {"Authorization": f"Bearer {access_token}"}
    tracks = []

    while tracks_url:
        response = requests.get(tracks_url, headers=headers)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch tracks: {response.status_code}")

        data = response.json()
        for item in data.get("items", []):
            track = item.get("track")
            if track:  # Skip if track is None (e.g., deleted tracks)
                tracks.append(
                    {
                        "name": track.get("name"),
                        "artist": track.get("artists", [{}])[0].get(
                            "name"
                        ),  # First artist
                        "album": track.get("album", {}).get("name"),
                        "uri": track.get("uri"),
                        "duration_ms": track.get("duration_ms"),
                        "release_date": track.get("album", {}).get("release_date"),
                    }
                )
        tracks_url = data.get("next")  # Move to next page if available

    return tracks


def process_playlists(access_token: str):
    """Main function to fetch and process playlists and their tracks.

    Args:
        access_token (str): The access token for the authenticated user.
    """
    try:
        # Fetch all playlists
        playlists = get_playlists(access_token)

        # Process each playlist
        for i, playlist in enumerate(playlists):
            playlist_name = playlist.get("name")
            tracks_url = playlist.get("tracks", {}).get("href")

            if tracks_url:
                tracks = get_playlist_tracks(tracks_url, access_token)
                print(f"\nPlaylist: {playlist_name}")
                for track in tracks:
                    print(
                        f"  - {track['name']} by {track['artist']} ({track['album']})"
                    )
            else:
                print(f"\nPlaylist: {playlist_name} (No tracks URL found)")

            # REMOVE AFTER TESTING
            if i > 3:
                break
            # REMOVE AFTER TESTING

    except Exception as e:
        print(f"Error: {e}")

In [12]:
import urllib.parse


def get_authorization_url(client_id, redirect_uri, scope="playlist-read-private"):
    """Generate the Spotify OAuth authorization URL."""
    params = {
        "response_type": "code",
        "client_id": client_id,
        "scope": scope,
        "redirect_uri": redirect_uri,
        "state": "random_state_string",  # Optional but recommended for CSRF protection
    }
    return f"https://accounts.spotify.com/authorize?{urllib.parse.urlencode(params)}"

In [ ]:
settings = get_settings()
redirect_uri = "https://127.0.0.1:8888/callback"
print(get_authorization_url(settings.SPOTIFY_CLIENT_ID, redirect_uri))

In [5]:
import base64


def get_access_token(client_id, client_secret, code, redirect_uri):
    """Exchange the authorization code for an access token.

    Parameters:
        client_id (str): Your Spotify Client ID.
        client_secret (str): Your Spotify Client Secret.
        code (str): The authorization code from the redirect URL.
        redirect_uri (str): The redirect URI registered in your Spotify app.

    Returns:
        dict: A dictionary containing the access token and other details
        (e.g., `access_token`, `token_type`, `expires_in`, etc.).
    """
    # Step 1: Prepare the Authorization header (Basic Auth)
    auth_string = f"{client_id}:{client_secret}"
    auth_bytes = auth_string.encode("utf-8")
    auth_base64 = base64.b64encode(auth_bytes).decode("utf-8")
    headers = {
        "Authorization": f"Basic {auth_base64}",
        "Content-Type": "application/x-www-form-urlencoded",
    }

    # Step 2: Prepare the request body
    data = {
        "grant_type": "authorization_code",
        "code": code,
        "redirect_uri": redirect_uri,
    }

    # Step 3: Send the POST   # e.g., "AQDiF5y6q8..."request
    token_url = "https://accounts.spotify.com/api/token"
    response = requests.post(token_url, headers=headers, data=data)

    # Step 4: Handle the response
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(
            f"Failed to get access token. Status Code: {response.status_code}, \
Response: {response.text}"
        )

In [ ]:
code = ""
redirect_uri = "https://127.0.0.1:8888/callback"

# Get the access token
token_data = get_access_token(
    settings.SPOTIFY_CLIENT_ID, settings.SPOTIFY_CLIENT_SECRET, code, redirect_uri
)
access_token = token_data["access_token"]
print("Access Token:", access_token)

In [15]:
playlists = get_playlists(access_token)

In [18]:
process_playlists(access_token)


Playlist: Weather Systems setlist Birmingham 2025
  - Deep - Remastered by Anathema (Judgement (Remastered))
  - Still Lake by Weather Systems (Ocean Without A Shore)
  - Synaesthesia by Weather Systems (Ocean Without A Shore)
  - Do Angels Sing Like Rain? by Weather Systems (Ocean Without A Shore)
  - Springfield by Anathema (The Optimist)
  - Are You There? Part 2 by Weather Systems (Ocean Without A Shore)
  - The Lost Song Part 3 by Anathema (Distant Satellites (Tour Edition))
  - A Simple Mistake by Anathema (We're Here Because We're Here)
  - Closer by Anathema (Original Album Classics)
  - Ocean Without A Shore by Weather Systems (Ocean Without A Shore)
  - Untouchable Part 1 by Anathema (Weather Systems)
  - Untouchable Part 2 by Anathema (Weather Systems)
  - Untouchable Part 3 by Weather Systems (Ocean Without A Shore)
  - Fragile Dreams by Anathema (Alternative 4)

Playlist: Steven Wilson
  - Vermillioncore by Steven Wilson (4 1/2)

Playlist: Steven Wilson Setlist Birmingham